# 🌊 瓦磘溝水質影像辨識 — YOLOv8 訓練 Notebook

**瓦磘溝行動研究社 × AI 影像辨識**

這份 Notebook 會帶你完成：
1. 環境設定（安裝套件、確認 GPU）
2. 下載並整理資料集
3. 訓練 YOLOv8 水質分類模型
4. 分析訓練結果（準確率、混淆矩陣、誤判案例）
5. 儲存模型到 Google Drive

---

### 📌 使用說明
**不需要寫任何程式！** 只要依序點擊每個灰色方塊左側的 ▶ 執行鈕即可。

### ⚡ 開始前請先切換 GPU
上方選單 → **執行階段** → **變更執行階段類型** → 選 **T4 GPU** → 儲存


---
## 第一步：確認 GPU 並安裝套件

執行這個步驟，系統會確認你的 GPU 是否已開啟，並安裝訓練需要的程式庫。

In [ ]:
# 確認 GPU 狀態
!nvidia-smi
import torch
print(f'\nPyTorch 版本：{torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU 已啟用：{torch.cuda.get_device_name(0)}')
else:
    print('❌ 未偵測到 GPU，請回到上方說明切換執行階段類型')

In [ ]:
# 安裝所需套件
!pip install ultralytics gdown -q
print('✅ 套件安裝完成')

---
## 第二步：掛載 Google Drive

這個步驟會要求你登入 Google 帳號，授權 Colab 存取你的 Google Drive。
訓練完成的模型會自動儲存回你的 Google Drive。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive 已掛載')

---
## 第三步：下載並整理資料集

從 Google 雲端硬碟下載瓦磘溝水質照片，並整理成訓練用的格式。

**資料切分比例：**
- 訓練集（train）：70%
- 驗證集（val）：20%
- 測試集（test）：10%

In [ ]:
import gdown, shutil, random, re
from pathlib import Path

FOLDER_ID   = '12OOjkS7GilRcvaVafh7NJKWk70mA1pDD'
RAW_DIR     = Path('/content/raw_data')
DATASET_DIR = Path('/content/dataset')
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.2
SEED        = 42
LABEL_MAP   = {'1': 'dirty', '3': 'turbid', '5': 'clean'}
IMG_EXTS    = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.heic', '.heif'}

def is_image(p):
    name = p.name.lower()
    return any(ext in name for ext in IMG_EXTS)

def get_ext(p):
    m = re.search(r'\.(heic|heif|jpg|jpeg|png|bmp|webp)', p.name.lower())
    return ('.' + m.group(1)) if m else '.jpg'

# 下載
if not RAW_DIR.exists():
    print('📥 下載中（約需 1–3 分鐘）...')
    gdown.download_folder(
        f'https://drive.google.com/drive/folders/{FOLDER_ID}',
        output=str(RAW_DIR), quiet=False, use_cookies=False
    )
else:
    print(f'ℹ️  已有 {RAW_DIR}，跳過下載')

# 顯示下載結果
print('\n📂 各資料夾照片數量：')
label_zh = {'dirty': '髒', 'turbid': '混濁', 'clean': '乾淨'}
for folder_name, label in LABEL_MAP.items():
    matches = [d for d in RAW_DIR.iterdir() if d.is_dir() and d.name.startswith(folder_name)]
    if matches:
        imgs = [f for f in matches[0].rglob('*') if f.is_file() and is_image(f)]
        print(f'  {folder_name}（{label_zh[label]}）：{len(imgs)} 張')

In [ ]:
# 整理成 train / val / test 結構
random.seed(SEED)
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ['train', 'val', 'test']:
    for label in LABEL_MAP.values():
        (DATASET_DIR / split / label).mkdir(parents=True, exist_ok=True)

print('📊 資料切分結果（70% train / 20% val / 10% test）：\n')
for folder_name, label in LABEL_MAP.items():
    matches = [d for d in RAW_DIR.iterdir() if d.is_dir() and d.name.startswith(folder_name)]
    if not matches:
        print(f'✗ 找不到 {folder_name}')
        continue
    src_dir = matches[0]
    images = [f for f in src_dir.rglob('*') if f.is_file() and is_image(f)]
    random.shuffle(images)
    n = len(images)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    for i, img in enumerate(images[:n_train]):
        shutil.copy(img, DATASET_DIR / 'train' / label / f'{label}_{i:04d}{get_ext(img)}')
    for i, img in enumerate(images[n_train:n_train+n_val]):
        shutil.copy(img, DATASET_DIR / 'val' / label / f'{label}_{i:04d}{get_ext(img)}')
    for i, img in enumerate(images[n_train+n_val:]):
        shutil.copy(img, DATASET_DIR / 'test' / label / f'{label}_{i:04d}{get_ext(img)}')

    n_test = n - n_train - n_val
    print(f'  {label_zh[label]:4s}（{label}）：train {n_train} / val {n_val} / test {n_test}  （共 {n} 張）')

total = sum(1 for _ in DATASET_DIR.rglob('*') if _.is_file())
print(f'\n✅ 共 {total} 張，資料集準備完成！')

In [ ]:
# 顯示各類別範例圖片（讓你確認照片內容正確）
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
titles = {'clean': '乾淨 (clean)', 'turbid': '混濁 (turbid)', 'dirty': '髒 (dirty)'}
colors = {'clean': '#2ecc71', 'turbid': '#f39c12', 'dirty': '#e74c3c'}

for row, label in enumerate(['clean', 'turbid', 'dirty']):
    imgs = list((DATASET_DIR / 'train' / label).glob('*'))[:3]
    for col, img_path in enumerate(imgs):
        ax = axes[row][col]
        try:
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, '讀取失敗', ha='center', va='center')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(titles[label], fontsize=12,
                          color=colors[label], fontweight='bold',
                          rotation=0, labelpad=80, va='center')

plt.suptitle('各類別範例照片（請確認照片內容正確）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/water_quality_model/sample_images.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✅ 範例圖片已儲存到 Google Drive')

---
## 第四步：訓練 YOLOv8 模型

### 為什麼用 YOLOv8 分類模型（而不是偵測模型）？

| | 物件偵測 | 影像分類 ✅ |
|--|---------|----------|
| 標註方式 | 需畫框框 | 資料夾分類即可 |
| 照片前處理 | 需找水面位置 | 已裁切好不需要 |
| 適合教學 | 概念較複雜 | 準確率直觀易懂 |

### 訓練參數說明

| 參數 | 值 | 原因 |
|------|----|------|
| model | yolov8s-cls | small 版本，準確率與速度均衡 |
| epochs | 100 | 搭配早停機制，不會過擬合 |
| imgsz | 224 | YOLOv8-cls 標準輸入尺寸 |
| batch | 32 | T4 GPU 記憶體足夠，訓練速度快 |
| patience | 20 | 20 個 epoch 沒進步就提早停止 |

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# 建立儲存資料夾
Path('/content/drive/MyDrive/water_quality_model').mkdir(parents=True, exist_ok=True)

# 訓練
model = YOLO('yolov8s-cls.pt')
results = model.train(
    data     = '/content/dataset',
    epochs   = 100,
    imgsz    = 224,
    batch    = 32,
    project  = '/content/runs',
    name     = 'water_quality',
    patience = 20,
    exist_ok = True,
)
print('\n🎉 訓練完成！')

---
## 第五步：查看訓練結果

In [ ]:
from IPython.display import Image as IPImage, display
from pathlib import Path

result_dir = Path('/content/runs/water_quality')
for img_name in ['results.png', 'confusion_matrix_normalized.png', 'confusion_matrix.png']:
    img_path = result_dir / img_name
    if img_path.exists():
        print(f'\n📈 {img_name}')
        display(IPImage(str(img_path)))

---
## 第六步：在測試集上完整評估

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import random

LABEL_ZH = {'clean': '乾淨', 'turbid': '混濁', 'dirty': '髒'}
model = YOLO('/content/runs/water_quality/weights/best.pt')

# 在 test 集推論
y_true, y_pred, y_conf, img_paths = [], [], [], []
labels = sorted([d.name for d in (DATASET_DIR / 'test').iterdir() if d.is_dir()])

for label in labels:
    for img_path in (DATASET_DIR / 'test' / label).glob('*'):
        r    = model(str(img_path))[0]
        pred = r.names[r.probs.top1]
        conf = r.probs.top1conf.item()
        y_true.append(label); y_pred.append(pred)
        y_conf.append(conf);  img_paths.append(img_path)

# 整體準確率
accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
print(f'\n✅ 測試集整體準確率：{accuracy:.1%}（{sum(t==p for t,p in zip(y_true,y_pred))}/{len(y_true)} 張）\n')

# 各類別指標
n = len(labels); idx = {l: i for i, l in enumerate(labels)}
cm = np.zeros((n, n), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[idx[t]][idx[p]] += 1

print(f'{"類別":10s} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print('-' * 44)
for i, label in enumerate(labels):
    tp = cm[i][i]; fp = cm[:,i].sum()-tp; fn = cm[i,:].sum()-tp
    p = tp/(tp+fp) if tp+fp>0 else 0
    r = tp/(tp+fn) if tp+fn>0 else 0
    f = 2*p*r/(p+r) if p+r>0 else 0
    print(f'{LABEL_ZH[label]:10s} {p:>10.1%} {r:>10.1%} {f:>10.1%}')

In [ ]:
# 混淆矩陣視覺化
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([f'{l}\n({LABEL_ZH[l]})' for l in labels], fontsize=11)
ax.set_yticklabels([f'{l}\n({LABEL_ZH[l]})' for l in labels], fontsize=11)
ax.set_xlabel('預測類別', fontsize=12); ax.set_ylabel('真實類別', fontsize=12)
ax.set_title('混淆矩陣（測試集）', fontsize=14)
for i in range(n):
    for j in range(n):
        color = 'white' if cm[i][j] > cm.max()/2 else 'black'
        ax.text(j, i, str(cm[i][j]), ha='center', va='center',
                fontsize=14, color=color, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/water_quality_model/confusion_matrix_test.png', dpi=150)
plt.show()
print('✅ 混淆矩陣已儲存到 Google Drive')

In [ ]:
# 顯示誤判案例
wrong = [(t, p, c, path) for t, p, c, path in zip(y_true, y_pred, y_conf, img_paths) if t != p]
print(f'\n誤判總數：{len(wrong)} 張（共 {len(y_true)} 張測試圖片）\n')

if wrong:
    sample = random.sample(wrong, min(6, len(wrong)))
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, (true, pred, conf, path) in zip(axes.flatten(), sample):
        try:
            img = Image.open(path).convert('RGB')
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, '無法讀取', ha='center', va='center')
        ax.set_title(f'真實：{LABEL_ZH[true]}\n預測：{LABEL_ZH[pred]}（{conf:.1%}）',
                     color='red', fontsize=10)
        ax.axis('off')
    for ax in axes.flatten()[len(sample):]:
        ax.axis('off')
    plt.suptitle('❌ 模型誤判案例', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/water_quality_model/wrong_cases.png', dpi=150)
    plt.show()
else:
    print('🎉 測試集全部預測正確！')

---
## 第七步：儲存模型到 Google Drive

In [ ]:
import shutil
from pathlib import Path

SAVE_DIR = Path('/content/drive/MyDrive/water_quality_model')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 複製模型
best_pt = Path('/content/runs/water_quality/weights/best.pt')
shutil.copy(best_pt, SAVE_DIR / 'best.pt')

# 複製訓練圖表
for f in Path('/content/runs/water_quality').glob('*.png'):
    shutil.copy(f, SAVE_DIR / f.name)

size_mb = (SAVE_DIR / 'best.pt').stat().st_size / 1024 / 1024
print(f'✅ 模型已儲存到 Google Drive')
print(f'   路徑：{SAVE_DIR}')
print(f'   模型大小：{size_mb:.1f} MB')

---
## 第八步（選用）：用新照片測試模型

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

LABEL_ZH = {'clean': '✅ 乾淨', 'turbid': '⚠️ 混濁', 'dirty': '❌ 髒'}
model = YOLO('/content/drive/MyDrive/water_quality_model/best.pt')

# 把你的照片上傳到 Colab 後，修改下方路徑
TEST_IMAGE = '/content/your_photo.jpg'   # ← 修改這裡

results = model(TEST_IMAGE)
r = results[0]
pred = r.names[r.probs.top1]
conf = r.probs.top1conf.item()

img = Image.open(TEST_IMAGE)
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.title(f'預測結果：{LABEL_ZH[pred]}\n信心度：{conf:.1%}', fontsize=14)
plt.axis('off')
plt.show()

print('各類別機率：')
for i, name in r.names.items():
    bar = '█' * int(r.probs.data[i].item() * 30)
    print(f'  {LABEL_ZH[name]:12s} {r.probs.data[i].item():5.1%}  {bar}')